# ViSTA-SLAM: Integration and Execution Guide

This notebook documents the environment configuration, compilation, and black-box validation of the ViSTA-SLAM pipeline within the isolated project ecosystem.

---

## 1. Environment Setup and Compilation

Execute the automated setup script from the repository root to initialize submodules, synchronize the virtual environment constraints, fetch pretrained weights from Hugging Face, and compile the native C++/CUDA extensions (`DBoW3Py` and `CuRoPE`).

In [1]:
%%bash
cd ..
chmod +x scripts/local/setup_vista_slam.sh
./scripts/local/setup_vista_slam.sh

=== Starting Setup of ViSTA-SLAM ===
=== Initializing Git Submodules ===
=== Synchronizing Virtual Environment ===


Resolved 220 packages in 1ms
Uninstalled 1 package in 1ms
 - dbow3py==0.0.3 (from file:///home/eze/AppliedFoundationModels/external/vista-slam/DBoW3Py)


=== Downloading Pretrained Weights ===


Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 471.83it/s]


✓ Downloaded
  path: /home/eze/AppliedFoundationModels/external/vista-slam/pretrains
=== Compiling DBoW3 C++ Bindings ===


Resolved 2 packages in 325ms
   Building dbow3py @ file:///home/eze/AppliedFoundationModels/external/vista-slam/DBoW3Py
      Built dbow3py @ file:///home/eze/AppliedFoundationModels/external/vista-slam/DBoW3Py
Prepared 1 package in 783ms
Installed 1 package in 0.96ms
 + dbow3py==0.0.3 (from file:///home/eze/AppliedFoundationModels/external/vista-slam/DBoW3Py)


=== Compiling CuRoPE CUDA Kernels ===
running build_ext


/home/eze/AppliedFoundationModels/.venv/lib/python3.12/site-packages/torch/utils/cpp_extension.py:497: UserWarning: Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
  warnings.warn(msg.format('we could not find ninja.'))
/home/eze/AppliedFoundationModels/.venv/lib/python3.12/site-packages/torch/utils/cpp_extension.py:416: UserWarning: The detected CUDA version (12.0) has a minor version mismatch with the version that was used to compile PyTorch (12.1). Most likely this shouldn't be a problem.
  warnings.warn(CUDA_MISMATCH_WARN.format(cuda_str_version, torch.version.cuda))
/home/eze/AppliedFoundationModels/.venv/lib/python3.12/site-packages/torch/utils/cpp_extension.py:426: UserWarning: There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.0
  warnings.warn(f'There are no {compiler_name} version bounds defined for CUDA version {cuda_str_version}')


copying build/lib.linux-x86_64-cpython-312/curope.cpython-312-x86_64-linux-gnu.so -> 
=== Completed Setup of ViSTA-SLAM ===


## 2. Generating the Custom Configuration File

Create the specialized `custom_3DRoomSearch.yaml` file inside the submodule's configuration directory.

In [2]:
%%writefile ../external/vista-slam/configs/custom_3DRoomSearch.yaml
# ViSTA-SLAM Custom Configuration
# For details about the params see comments in: https://github.com/zhangganlin/vista-slam/blob/main/configs/default.yaml

# ==============================================================================
# 3D ROOM SEARCH RELEVANT PARAMETERS
# ==============================================================================
# Paths (relative to repository root during execution)
STA_pretrain_path: external/vista-slam/pretrains/frontend_sta_weights.pth   
vocab_path: external/vista-slam/pretrains/ORBvoc.txt

# output folder to save results; can be overridden by the command-line argument --output
output_dir: data/vista_slam_output

keyframe_detection: "stride"
stride: 9
point_conf_thres: 4.2 
neighbor_edge_num: 2  

# ==============================================================================
# DEFAULT / NON-RELEVANT PARAMETERS
# ==============================================================================
device: "cuda"
verbose: False
random_seed: 43
max_view_num: 400
rerun_vis: False
rerun_url: rerun+http://127.0.0.1:9876/proxy
rerun_save: False
loop_edge_num: 2
loop_dist_min: 40     
loop_nms: 40          
loop_cand_thresh_neighbor: 5 
rel_pose_thres: 0.75
flow_thres: 5.0
pgo_every: 500

Overwriting ../external/vista-slam/configs/custom_3DRoomSearch.yaml


## 3 Dataset Acquisition (only for demonstration of ViSTA-SLAM in this notebook)

Download and extract the official TUM RGB-D `freiburg1_room` sequence directly into the shared project data directory.

In [ ]:
%%bash
cd ..
mkdir -p data/datasets

# Download the compressed dataset payload if it does not already exist
wget -q --show-progress -N https://vision.in.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_room.tgz -P data/datasets/

# Extract the sequence into the target directory
tar -xzf data/datasets/rgbd_dataset_freiburg1_room.tgz -C data/datasets/

### Dataset Cleanup

Remove unnecessary sensor modalities, depth maps, and log files to isolate only the RGB image stream required for the tracking sequence demonstration.

In [ ]:
%%bash
cd ..
rm -rf data/datasets/rgbd_dataset_freiburg1_room/depth \
       data/datasets/rgbd_dataset_freiburg1_room/depth.txt \
       data/datasets/rgbd_dataset_freiburg1_room/accelerometer.txt \
       data/datasets/rgbd_dataset_freiburg1_room/groundtruth.txt \
       data/datasets/rgbd_dataset_freiburg1_room/rgb.txt

## 4. Inference 

Execute the tracking sequence using our specialized configuration layout (`custom_3DRoomSearch.yaml`).

### 4.1 Inference (Blind) - a bit faster

This run operates without a live visualization frontend, maximizing processing throughput.

In [ ]:
%%bash
cd ..
uv run python external/vista-slam/run.py \
  --config external/vista-slam/configs/custom_3DRoomSearch.yaml \
  --images "data/datasets/rgbd_dataset_freiburg1_room/rgb/*.png" \
  --output data/vista_slam_output

### 4.2 Inference (Live Spatial Visualization) - nice to see what is going on

Execute the tracking sequence with live visualization enabled. This setup requires the Rerun viewer to be running concurrently in an independent terminal or background process (`uv run rerun`).

The next cell starts the Rerun viewer as a background subprocess. This ensures the viewer runs continuously without blocking the execution of subsequent notebook cells.

In [3]:
import subprocess
import time

# Spawn the Rerun viewer asynchronously
print("Starting Rerun server...")
rerun_process = subprocess.Popen(
    ["uv", "run", "rerun"],
    cwd="..",
    stdout=subprocess.DEVNULL, 
    stderr=subprocess.DEVNULL
)

# Allow the server a brief moment to bind to the gRPC ports
time.sleep(3)
print(f"Rerun active (PID: {rerun_process.pid}). Proceed to the next cell.")

Starting Rerun server...
Rerun active (PID: 735763). Proceed to the next cell.


The next cell invokes ViSTA-SLAM with visualization active. It streams camera poses, keyframe topologies, and reconstructed spatial point clouds to the Rerun viewer spawned in the previous step.

In [4]:
%%bash
cd ..
uv run python external/vista-slam/run.py \
  --config external/vista-slam/configs/custom_3DRoomSearch.yaml \
  --images "data/datasets/rgbd_dataset_freiburg1_room/rgb/*.png" \
  --output data/vista_slam_output \
  --vis

[PoseGraphOpt] Pose graph optimization (at keyframe 152) ...


[Progress] 100%|████████            62 frames]██| [1362/1362 frames]

[PoseGraphOpt] Pose graph optimization done.


[Progress] 100%|██████████| [1362/1362 frames]


[INFO] Total keyframes detected: 152
[INFO] Total time spent: 29.4 s
[INFO] Saving data to data/vista_slam_output ... Done.


## 5. ViSTA-SLAM Outputs

ViSTA-SLAM serializes the finalized trajectory, dense depth maps, pose graph, and the reconstructed 3D point cloud directly into the designated output directory upon completion. 

![ViSTA-SLAM Output Contents](images/vista_slam_output_snapshot.png)

#### [Optional] Fallback: Download Pre-computed ViSTA-SLAM Output

Bypass the execution and download the pre-computed optimization results to proceed directly with the visualization.

(Same results you would get if performing inference with the given config file and data above)

In [5]:
%%bash
cd ..
mkdir -p data
# Download the pre-computed archive and extract to mimic inference results
wget -q --show-progress -O data/vista_slam_notebook_tum_results.tar.gz "https://huggingface.co/datasets/ezeschwarzbock/vista-slam-notebook-tum-results/resolve/main/vista_slam_notebook_tum_results.tar.gz?download=true"
tar -xzvf data/vista_slam_notebook_tum_results.tar.gz -C data/


     0K .......... .......... .......... .......... ..........  0% 4.86M 27s
    50K .......... .......... .......... .......... ..........  0% 5.37M 25s
   100K .......... .......... .......... .......... ..........  0% 10.4M 21s
   150K .......... .......... .......... .......... ..........  0% 5.37M 22s
   200K .......... .......... .......... .......... ..........  0% 5.33M 22s
   250K .......... .......... .......... .......... ..........  0% 5.19M 23s
   300K .......... .......... .......... .......... ..........  0% 5.32M 23s
   350K .......... .......... .......... .......... ..........  0% 4.65M 24s
   400K .......... .......... .......... .......... ..........  0% 5.24M 24s
   450K .......... .......... .......... .......... ..........  0% 4.05M 24s
   500K .......... .......... .......... .......... ..........  0% 5.71M 24s
   550K .......... .......... .......... .......... ..........  0% 4.74M 24s
   600K .......... .......... .......... .......... ..........  0% 5.32M 24

vista_slam_output/
vista_slam_output/images.npy
vista_slam_output/trajectory.npy
vista_slam_output/scales.npy
vista_slam_output/pointcloud.ply
vista_slam_output/depths.npy
vista_slam_output/confs.npz
vista_slam_output/view_graph.npz
vista_slam_output/intrinsics.npy


### 5.1 Visualize Optimization Results

Execute the offline visualization script to inspect the fused 3D point cloud, optimized camera poses, and the resulting pose graph.

In [6]:
%%bash
cd ..
uv run python external/vista-slam/scripts/vis_slam_results.py data/vista_slam_output

Loaded point cloud: data/vista_slam_output/pointcloud.ply with 3874541 points
Loaded 152 camera poses from data/vista_slam_output/trajectory.npy
Loaded and visualizing view graph from data/vista_slam_output/view_graph.npz


## 6. TODO: Backprojection of an object mask with ViSTA-SLAM Outputs